# One Generic Model vs Specialist Models

Is a single **generic** model (trained on every grade) as good as using a
**specialist** model for each part of the grade range? We compare them the fair way
— each generic-vs-specialist match-up is judged **only on the grades that specialist
is built for**.

| Model | What it is | Built for |
|-------|-----------|-----------|
| **Generic** | one GBT trained on every grade | all grades |
| **Specialist (Grades 1–8, 10–12)** | a GBT trained only on those grades | the continuing grades |
| **Specialist (K & 9)** | an XGBoost trained only on K & 9 | the entry grades |

**The two fair comparisons**
- On **Grades 1–8 & 10–12**: Generic vs the Grades-1–8/10–12 Specialist.
- On **Kindergarten & Grade 9**: Generic vs the K-&-9 Specialist.

"Current method (CSR)" is shown alongside as the approach used today, and the
generic model's own results are shown separately at the end of each stage.

> Uses the same training, metrics and baseline as the main pipeline
> (`CPS_Enrollment_Pipeline.ipynb`). Held-out year first, then a multi-year backtest
> for a steadier read.

## Stage 1 — Setup

### 1.1 Imports & configuration

Same tables, year windows, GBT/XGBoost hyperparameters and feature sets as the main
pipeline. GBT features exclude `IS_MIGRANT_ANOMALY_YEAR` (it drives the weighting,
not a feature) and use `GRADE` via a `StringIndexer`; the XGBoost feature set keeps it.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.sql.window import Window

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, r2_score

FEATURE_TABLE = "ml_enrollment_feature_table_v4"
LABEL_COL  = "ENROLLMENT"
TIME_COL   = "SCHOOL_YEAR"
WEIGHT_COL = "sample_weight"

TRAIN_START_YEAR = 2020
TRAIN_END_YEAR   = 2025
VAL_START_YEAR   = 2026
VAL_END_YEAR     = 2026
ENTRY_GRADES     = ["K", "9"]

REPUTATION_TABLE   = "ml_reputation_score"
REPUTATION_PILLARS = [
    "EFFECTIVE_LEADERS", "COLLABORATIVE_TEACHERS", "INVOLVED_FAMILIES",
    "SUPPORTIVE_ENVIRONMENT", "AMBITIOUS_INSTRUCTION",
]
REPUTATION_FEATURES = [f"{p}_LAST_YEAR" for p in REPUTATION_PILLARS]

# GBT hyperparameters — same fixed params as the main pipeline
BEST_MAX_DEPTH = 5
BEST_MAX_ITER  = 100
BEST_STEP_SIZE = 0.1
BEST_SUBSAMPLING_RATE = 0.8

# XGBoost hyperparameters
XGB_PARAMS = dict(
    n_estimators=100, max_depth=5, learning_rate=0.1, subsample=0.8,
    colsample_bytree=0.8, objective="reg:squarederror", tree_method="hist",
    missing=np.nan, random_state=42, n_jobs=-1,
)

_BASE = [
    "SAME_GRADE_LAST_YEAR", "SAME_GRADE_2YR_AGO",
    "FEEDER_GRADE_LAST_YEAR", "FEEDER_GRADE_2YR_AGO",
    "SCHOOL_TOTAL_LAST_YEAR", "COHORT_SURVIVAL_RATE", "AVG_SURVIVAL_RATE_3YR",
    "DISTRICT_GRADE_ENROLLMENT_LAST_YEAR",
    "EFFECTIVE_LEADERS_LAST_YEAR", "COLLABORATIVE_TEACHERS_LAST_YEAR",
    "INVOLVED_FAMILIES_LAST_YEAR", "SUPPORTIVE_ENVIRONMENT_LAST_YEAR",
    "AMBITIOUS_INSTRUCTION_LAST_YEAR",
]
_TREND = [
    "SAME_GRADE_AVG_3YR", "SAME_GRADE_TREND", "FEEDER_GRADE_TREND",
    "SCHOOL_TOTAL_2YR_AGO", "SCHOOL_TOTAL_AVG_3YR", "SCHOOL_TOTAL_TREND",
    "DISTRICT_GRADE_ENROLLMENT_2YR_AGO", "DISTRICT_GRADE_ENROLLMENT_AVG_3YR",
    "DISTRICT_GRADE_ENROLLMENT_TREND",
]
GBT_FEATURE_CANDIDATES = _BASE + _TREND
XGB_FEATURE_CANDIDATES = _BASE + ["IS_MIGRANT_ANOMALY_YEAR"] + _TREND
NULL_FILL_CANDIDATES = [
    "SAME_GRADE_2YR_AGO", "FEEDER_GRADE_LAST_YEAR", "FEEDER_GRADE_2YR_AGO",
    "COHORT_SURVIVAL_RATE", "AVG_SURVIVAL_RATE_3YR", "DISTRICT_GRADE_ENROLLMENT_LAST_YEAR",
    "SCHOOL_TOTAL_LAST_YEAR",
    "EFFECTIVE_LEADERS_LAST_YEAR", "COLLABORATIVE_TEACHERS_LAST_YEAR",
    "INVOLVED_FAMILIES_LAST_YEAR", "SUPPORTIVE_ENVIRONMENT_LAST_YEAR",
    "AMBITIOUS_INSTRUCTION_LAST_YEAR",
] + _TREND
print("Configuration loaded.")

### 1.2 Load feature table & reputation features

Loads the feature table. If it already carries the five 5Essentials pillar features
(built by the main pipeline) they are used **as-is**, so every metric matches the
main notebook exactly. Only if they are absent are they re-derived from
`ml_reputation_score` with the *same* logic as the main pipeline's Stage 1.7
(dedup → +1-year lag → backward-only per-school imputation), keeping the notebook
self-contained.

In [ ]:
sdf = spark.read.table(FEATURE_TABLE)

if all(c in sdf.columns for c in REPUTATION_FEATURES):
    print("5Essentials reputation already in the table - using it as-is (matches the main pipeline).")
else:
    print("5Essentials reputation absent - re-deriving from ml_reputation_score (self-contained).")
    sdf = sdf.drop(*[c for c in REPUTATION_FEATURES if c in sdf.columns])
    rep_raw = spark.read.table(REPUTATION_TABLE)
    missing_rep = {"SCHOOL_KEY", "SCHOOL_YEAR", *REPUTATION_PILLARS} - set(rep_raw.columns)
    if missing_rep:
        raise ValueError(f"{REPUTATION_TABLE} is missing required columns: {missing_rep}")

    rep_dedup = (
        rep_raw
        .filter(F.col("SCHOOL_KEY").isNotNull() & F.col("SCHOOL_YEAR").isNotNull())
        .groupBy(F.col("SCHOOL_KEY").cast("long").alias("SCHOOL_KEY"),
                 F.col("SCHOOL_YEAR").cast("int").alias("SCHOOL_YEAR"))
        .agg(*[F.avg(F.col(p).cast("double")).alias(p) for p in REPUTATION_PILLARS])
    )
    rep_lag = rep_dedup.select(
        F.col("SCHOOL_KEY").alias("_rep_school"),
        (F.col("SCHOOL_YEAR") + 1).alias("_rep_join_year"),
        *[F.col(p).alias(f"{p}_LAST_YEAR") for p in REPUTATION_PILLARS],
    )
    _before = sdf.count()
    sdf = (
        sdf.join(rep_lag,
                 on=[sdf["SCHOOL_KEY"].cast("long") == rep_lag["_rep_school"],
                     sdf[TIME_COL].cast("int") == rep_lag["_rep_join_year"]],
                 how="left")
        .drop("_rep_school", "_rep_join_year")
    )
    assert sdf.count() == _before, "Reputation join changed the row count (fan-out!)"

    rep_sy = sdf.groupBy("SCHOOL_KEY", TIME_COL).agg(
        *[F.max(F.col(feat)).alias(feat) for feat in REPUTATION_FEATURES]
    )
    w_rep_back = (Window.partitionBy("SCHOOL_KEY").orderBy(TIME_COL)
                  .rowsBetween(Window.unboundedPreceding, -1))
    for feat in REPUTATION_FEATURES:
        rep_sy = rep_sy.withColumn(feat, F.coalesce(F.col(feat), F.avg(F.col(feat)).over(w_rep_back)))
    sdf = sdf.drop(*REPUTATION_FEATURES).join(rep_sy, on=["SCHOOL_KEY", TIME_COL], how="left")

print(f"Feature table rows: {sdf.count():,}")

### 1.3 Feature lists & pandas copy

The Spark frame feeds the GBT models; a pandas copy (nulls kept) feeds the XGBoost
model and the CSR baseline. Feature lists are filtered to the columns present.

In [ ]:
avail = set(sdf.columns)
FEATURE_COLS   = [c for c in GBT_FEATURE_CANDIDATES if c in avail]   # GBT inputs (+ GRADE_idx)
XGB_FEATURES   = [c for c in XGB_FEATURE_CANDIDATES if c in avail]
NULL_FILL_COLS = [c for c in NULL_FILL_CANDIDATES if c in avail]
FLOAT_COLS = {f.name for f in sdf.schema.fields if str(f.dataType) in ("DoubleType()", "FloatType()")}
missing = [c for c in XGB_FEATURE_CANDIDATES if c not in avail]
if missing:
    print("Note - not in the table yet (re-run main Stage 1):", missing)
print(f"GBT features: {len(FEATURE_COLS)}  |  XGB features: {len(XGB_FEATURES)}")

xcols = list(dict.fromkeys(
    XGB_FEATURES + [LABEL_COL, TIME_COL, "GRADE", "SCHOOL_KEY",
                    "FEEDER_GRADE_LAST_YEAR", "AVG_SURVIVAL_RATE_3YR"]
))
pdf = sdf.select(*xcols).toPandas()
for c in XGB_FEATURES + [LABEL_COL, TIME_COL, "FEEDER_GRADE_LAST_YEAR", "AVG_SURVIVAL_RATE_3YR"]:
    pdf[c] = pd.to_numeric(pdf[c], errors="coerce")
pdf["GRADE"] = pdf["GRADE"].astype(str)

is_entry = pdf["GRADE"].isin(ENTRY_GRADES)
print(f"Rows: {len(pdf):,}  |  K&9: {int(is_entry.sum()):,}  |  continuing: {int((~is_entry).sum()):,}")
print("Years:", sorted(int(y) for y in pdf[TIME_COL].dropna().unique()))

### 1.4 Helpers

The GBT path mirrors the main pipeline exactly (train-only median imputation, migrant
down-weighting, NaN/Inf cleaning, and the `StringIndexer → VectorAssembler →
GBTRegressor` pipeline). The XGBoost path keeps nulls. Every model is scored with the
**same metric helper as the main pipeline** so the comparison is apples-to-apples.

In [ ]:
def median_fills_from(train_sdf):
    """Train-only medians for the null-fillable columns (main pipeline's impute logic)."""
    fills = {}
    for c in NULL_FILL_COLS:
        q = train_sdf.filter(F.col(c).isNotNull()).approxQuantile(c, [0.5], 0.01)
        fills[c] = q[0] if q else 0.0
    return fills


def clean_df(sdf, float_cols, feature_cols):
    """Drop rows with NaN/Inf (floats) or NULL (non-floats) in any feature column (verbatim from main)."""
    out = sdf
    for c in feature_cols:
        if c in float_cols:
            out = out.filter(~F.isnan(c) & (F.col(c) != float("inf")) & (F.col(c) != float("-inf")) & F.col(c).isNotNull())
        else:
            out = out.filter(F.col(c).isNotNull())
    return out


def gbt_pipeline():
    idx = StringIndexer(inputCol="GRADE", outputCol="GRADE_idx", handleInvalid="keep")
    asm = VectorAssembler(inputCols=FEATURE_COLS + ["GRADE_idx"], outputCol="features", handleInvalid="keep")
    gbt = GBTRegressor(
        labelCol=LABEL_COL, featuresCol="features", weightCol=WEIGHT_COL,
        seed=42, maxBins=256, maxDepth=BEST_MAX_DEPTH, maxIter=BEST_MAX_ITER,
        stepSize=BEST_STEP_SIZE, subsamplingRate=BEST_SUBSAMPLING_RATE,
    )
    return Pipeline(stages=[idx, asm, gbt])


def train_gbt(train_sdf):
    """Main pipeline's GBT recipe: train-only median fill -> migrant weights -> clean -> fit."""
    fills = median_fills_from(train_sdf)
    tr = train_sdf.fillna(fills).withColumn(
        WEIGHT_COL, F.when(F.col("IS_MIGRANT_ANOMALY_YEAR") == 1, F.lit(0.3)).otherwise(F.lit(1.0))
    )
    tr = clean_df(tr, FLOAT_COLS, FEATURE_COLS)
    return gbt_pipeline().fit(tr), fills


def gbt_frame(model, fills, test_sdf):
    """Score a Spark test set (median-filled with the model's train medians); to pandas."""
    return (
        model.transform(test_sdf.fillna(fills))
        .select("GRADE",
                F.col(LABEL_COL).cast("double").alias("y"),
                F.col("prediction").cast("double").alias("yhat"))
        .toPandas()
    )


def fit_xgb(train_pdf):
    weights = np.where(train_pdf["IS_MIGRANT_ANOMALY_YEAR"].fillna(0) == 1, 0.3, 1.0)
    model = xgb.XGBRegressor(**XGB_PARAMS)
    model.fit(train_pdf[XGB_FEATURES], train_pdf[LABEL_COL].astype(float), sample_weight=weights)
    return model


def xgb_frame(model, test_pdf):
    return pd.DataFrame({
        "GRADE": test_pdf["GRADE"].astype(str).values,
        "y": test_pdf[LABEL_COL].astype(float).values,
        "yhat": np.clip(model.predict(test_pdf[XGB_FEATURES]), 0, None),
    })


def csr_frame(test_pdf):
    # CSR baseline = FEEDER_GRADE_LAST_YEAR * coalesce(AVG_SURVIVAL_RATE_3YR, 1.0)  (verbatim from main)
    return pd.DataFrame({
        "GRADE": test_pdf["GRADE"].astype(str).values,
        "y": test_pdf[LABEL_COL].astype(float).values,
        "yhat": (test_pdf["FEEDER_GRADE_LAST_YEAR"] * test_pdf["AVG_SURVIVAL_RATE_3YR"].fillna(1.0)).values,
    })


def pandas_metrics(y, y_hat):
    """MAE, RMSE, MAPE, MedAPE and R2 — verbatim from the main pipeline (Stage 3.1)."""
    y = np.asarray(y, float)
    y_hat = np.asarray(y_hat, float)
    pos = y > 0
    return {
        "MAE": mean_absolute_error(y, y_hat),
        "RMSE": float(np.sqrt(np.mean((y - y_hat) ** 2))),
        "MAPE": float(np.mean(np.abs((y[pos] - y_hat[pos]) / y[pos])) * 100),
        "MedAPE": float(np.median(np.abs((y[pos] - y_hat[pos]) / y[pos])) * 100),
        "R2": r2_score(y, y_hat) if len(np.unique(y)) > 1 else np.nan,
    }


def compare(frames, grade_filter):
    """frames: {label: dataframe with GRADE,y,yhat}. Same metrics as the main pipeline, per segment."""
    rows = {}
    for name, fr in frames.items():
        sub = fr[fr["GRADE"].map(grade_filter)]
        if len(sub):
            rows[name] = {"n": int(len(sub)), **pandas_metrics(sub["y"].values, sub["yhat"].values)}
    return pd.DataFrame(rows).T[["n", "MAE", "RMSE", "MAPE", "MedAPE", "R2"]]


CONTINUING = lambda g: g not in ENTRY_GRADES
K9 = lambda g: g in ENTRY_GRADES


def by_segment(frame):
    """One model's results split into the two grade groups (shown on its own)."""
    rows = {}
    for label, keep in [("Grades 1-8 & 10-12", CONTINUING), ("Kindergarten & Grade 9", K9)]:
        sub = frame[frame["GRADE"].map(keep)]
        if len(sub):
            rows[label] = {"n": int(len(sub)), **pandas_metrics(sub["y"].values, sub["yhat"].values)}
    return pd.DataFrame(rows).T[["n", "MAE", "RMSE", "MAPE", "MedAPE", "R2"]]


PLOT_NAMES = {"MAE": "Average error (students)", "MAPE": "Average % error", "MedAPE": "Typical % error"}

def plot_models(tbl, title):
    """Simple bar chart: one bar per model, three plain-language error measures."""
    colors = ["#2e86ab", "#7DBA84", "#d1495b"][:len(tbl)]
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
    for ax, m in zip(axes, ["MAE", "MAPE", "MedAPE"]):
        bars = ax.bar(range(len(tbl)), tbl[m], color=colors)
        ax.bar_label(bars, fmt="%.0f", padding=2, fontsize=9)
        ax.set_title(PLOT_NAMES[m])
        ax.set_xticks(range(len(tbl)))
        ax.set_xticklabels(tbl.index, rotation=0)
    fig.suptitle(title + "  (shorter bars are better)", fontweight="bold")
    fig.tight_layout(); plt.show()

## Stage 2 — Held-out year

Train on the earlier years, then check accuracy on the held-out year.

### 2.1 Train the models

In [ ]:
tr_sdf = sdf.filter((F.col(TIME_COL) >= TRAIN_START_YEAR) & (F.col(TIME_COL) <= TRAIN_END_YEAR))
va_sdf = sdf.filter((F.col(TIME_COL) >= VAL_START_YEAR) & (F.col(TIME_COL) <= VAL_END_YEAR))
is_e = F.col("GRADE").isin(ENTRY_GRADES)

generic_model, generic_fills     = train_gbt(tr_sdf)                # Generic: GBT, every grade
spec_cont_model, spec_cont_fills = train_gbt(tr_sdf.filter(~is_e))  # Specialist: GBT, grades 1-8, 10-12
spec_k9_model = fit_xgb(pdf[(pdf[TIME_COL] >= TRAIN_START_YEAR) & (pdf[TIME_COL] <= TRAIN_END_YEAR) & is_entry])  # Specialist: XGBoost, K & 9
print("Trained: Generic (GBT, all grades), Specialist 1-8/10-12 (GBT), Specialist K&9 (XGBoost).")

# Predictions on the held-out year (all grades scored by each model)
va_pdf = pdf[(pdf[TIME_COL] >= VAL_START_YEAR) & (pdf[TIME_COL] <= VAL_END_YEAR)]
generic_pred   = gbt_frame(generic_model, generic_fills, va_sdf)
spec_cont_pred = gbt_frame(spec_cont_model, spec_cont_fills, va_sdf)
spec_k9_pred   = xgb_frame(spec_k9_model, va_pdf)
current_pred   = csr_frame(va_pdf)
print(f"Scored {len(generic_pred):,} rows in the held-out year.")

### 2.2 Grades 1–8 & 10–12 — Generic vs Specialist

Judged only on these grades.

In [ ]:
cont = compare({
    "Generic": generic_pred,
    "Specialist": spec_cont_pred,
    "Current (CSR)": current_pred,
}, CONTINUING)
print("Grades 1-8 & 10-12 (held-out year)")
print(cont.round(2).to_string())
plot_models(cont, "Grades 1-8 & 10-12")

### 2.3 Kindergarten & Grade 9 — Generic vs Specialist

Judged only on K and Grade 9 (and each on its own).

In [ ]:
k9 = compare({
    "Generic": generic_pred,
    "Specialist": spec_k9_pred,
    "Current (CSR)": current_pred,
}, K9)
print("Kindergarten & Grade 9 (held-out year)")
print(k9.round(2).to_string())
plot_models(k9, "Kindergarten & Grade 9")

for g, name in [("K", "Kindergarten"), ("9", "Grade 9")]:
    one = (lambda gg: lambda x: x == gg)(g)
    t = compare({"Generic": generic_pred, "Specialist": spec_k9_pred, "Current (CSR)": current_pred}, one)
    if len(t):
        print(f"\n{name} only")
        print(t.round(2).to_string())

### 2.4 The generic model on its own

The single generic model's results for each grade group, shown separately.

In [ ]:
print("Generic model - its own results by grade group (held-out year)")
print(by_segment(generic_pred).round(2).to_string())

## Stage 3 — Multi-year backtest

The held-out year alone is a small sample (especially for K & 9), so we repeat the
test across several past years and pool the results for a steadier read.

### 3.1 Run the backtest

In [ ]:
BACKTEST_START_YEAR = TRAIN_START_YEAR + 3
data_max = int(pdf[TIME_COL].dropna().max())
EVAL_YEARS = list(range(BACKTEST_START_YEAR, data_max + 1))
print("Years tested:", EVAL_YEARS)

gen_parts, sc_parts, sk_parts, cur_parts = [], [], [], []
for T in EVAL_YEARS:
    past_sdf = sdf.filter(F.col(TIME_COL) < T)
    cur_sdf  = sdf.filter(F.col(TIME_COL) == T)
    if not cur_sdf.take(1) or not past_sdf.take(1):
        continue
    gm, gf = train_gbt(past_sdf)
    sm, sf = train_gbt(past_sdf.filter(~F.col("GRADE").isin(ENTRY_GRADES)))
    cur_pdf = pdf[pdf[TIME_COL] == T]
    past_entry = pdf[(pdf[TIME_COL] < T) & pdf["GRADE"].isin(ENTRY_GRADES)]

    gen_parts.append(gbt_frame(gm, gf, cur_sdf))
    sc_parts.append(gbt_frame(sm, sf, cur_sdf))
    if not past_entry.empty:
        sk_parts.append(xgb_frame(fit_xgb(past_entry), cur_pdf))
    cur_parts.append(csr_frame(cur_pdf))
    print(f"  Year {T}: done")

generic_bt   = pd.concat(gen_parts, ignore_index=True)
spec_cont_bt = pd.concat(sc_parts, ignore_index=True)
spec_k9_bt   = pd.concat(sk_parts, ignore_index=True)
current_bt   = pd.concat(cur_parts, ignore_index=True)
print(f"Pooled {len(generic_bt):,} rows across all tested years.")

### 3.2 The two fair comparisons (all years pooled)

In [ ]:
cont = compare({"Generic": generic_bt, "Specialist": spec_cont_bt, "Current (CSR)": current_bt}, CONTINUING)
print("Grades 1-8 & 10-12 (all tested years)")
print(cont.round(2).to_string())
plot_models(cont, "Grades 1-8 & 10-12 (all years)")

k9 = compare({"Generic": generic_bt, "Specialist": spec_k9_bt, "Current (CSR)": current_bt}, K9)
print("\nKindergarten & Grade 9 (all tested years)")
print(k9.round(2).to_string())
plot_models(k9, "Kindergarten & Grade 9 (all years)")

for g, name in [("K", "Kindergarten"), ("9", "Grade 9")]:
    one = (lambda gg: lambda x: x == gg)(g)
    t = compare({"Generic": generic_bt, "Specialist": spec_k9_bt, "Current (CSR)": current_bt}, one)
    if len(t):
        print(f"\n{name} only")
        print(t.round(2).to_string())

### 3.3 Generic model on its own, and the verdict

In [ ]:
print("Generic model - its own results by grade group (all tested years)")
print(by_segment(generic_bt).round(2).to_string())

print("\nWho wins each group (smaller average error):")
print(f"  Grades 1-8 & 10-12 : {cont['MAE'].idxmin()}")
print(f"  Kindergarten & 9   : {k9['MAE'].idxmin()}")
print("\nHow to read it:")
print("  - 'Specialist' winning means a dedicated model for those grades is worth keeping.")
print("  - 'Generic' winning means one model for every grade is good enough there.")